# 07 — Figure Generation

Generate all publication-quality figures.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
from sklearn.model_selection import train_test_split
from src.config import *
from src.data_extraction import load_all_data
from src.preprocessing import preprocess_expression_matrix, prepare_ml_arrays
from src.training import run_model_comparison
from src.models import get_model_definitions, get_endosomal_features
from src.interpretability import compute_shap_values, shap_feature_importance
from src.models import stability_selection
from src.figures import *
set_publication_style()

In [ ]:
data = load_all_data()
matrix = preprocess_expression_matrix(data['mouse_matrix'])
X, y, names = prepare_ml_arrays(matrix)

In [ ]:
# Fig 1: Volcano
plot_volcano(data['sig_proteins'])
print('Fig 1 done')

In [ ]:
# Fig 2: ROC (needs trained results)
results = run_model_comparison(
    X, y, names,
    cv_config={'n_splits': 5, 'n_repeats': 3, 'seed': 42},
    n_stability_iter=100, verbose=False,
)
plot_roc_comparison(results, data_source=data['data_source'])
print('Fig 2 done')

In [ ]:
# Fig 4: Stability selection
from sklearn.preprocessing import StandardScaler
endo = get_endosomal_features(names)
endo_idx = [names.index(f) for f in endo if f in names]
X_sc = StandardScaler().fit_transform(X[:, endo_idx])
sel_probs = stability_selection(X_sc, y, n_iterations=200, sample_fraction=0.5)
plot_stability_selection(sel_probs, endo)
print('Fig 4 done')

In [ ]:
# Fig 3: SHAP
X_endo = X[:, endo_idx]
X_tr, X_te, y_tr, y_te = train_test_split(X_endo, y, test_size=0.3, random_state=42, stratify=y)
models = get_model_definitions()
xgb_pipe = models['Endosomal-XGB']
xgb_pipe.fit(X_tr, y_tr)
shap_vals, _ = compute_shap_values(xgb_pipe, X_tr, X_te, endo)
imp = shap_feature_importance(shap_vals, endo)
plot_shap_bar(imp)
print('Fig 3B done')

In [ ]:
# Fig S1: Correlation clustermap
from src.figures import plot_correlation_clustermap
plot_correlation_clustermap(matrix, feature_cols=endo)
print('Fig S1 done')

In [ ]:
# Fig S2: Forest plot
plot_model_comparison_forest(results)
print('Fig S2 done')
print(f'\nAll figures saved to {FIGURES_DIR}')